In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from pyspark.sql.types import *

In [0]:
data = [(1,"Ajay",60000),
        (2,"Ravi",70000),
        (3,"Mina",80000)]

schema = StructType([StructField("id",IntegerType(),True),
                     StructField("name",StringType(),True),
                     StructField("salary",IntegerType(),True)])
df = spark.createDataFrame(data,schema=schema)
display(df)

In [0]:
df.filter(F.col("salary")>=80000).display()

In [0]:
# Broadcast Join (Map-Side Join) in PySpark
#
# What it is:
#   Normally, Spark joins shuffle both tables so that rows with matching keys land on the
#   same executor (a "shuffle join" / sort-merge join). A broadcast join skips the shuffle:
#   the SMALLER table is copied ("broadcast") to every executor, and the larger table's
#   partitions are joined locally against that in-memory copy.
#
# Why it is faster:
#   - No shuffle stage = no network transfer / disk I/O for the big table
#   - No sorting required; joins happen per row against the small in-memory table
#   - Works great for joining a large fact table with small dimension/lookup tables
#
# When Spark does it automatically:
#   - If the smaller side is estimated under spark.sql.autoBroadcastJoinThreshold
#     (default 10 MB), Spark's cost-based optimizer chooses broadcast on its own.
#
# How to force it manually:
#   - Wrap the small DataFrame with F.broadcast(...) in the join

# --- Demo using the small df from earlier as the "dimension" table ---
dept_data = [(1, "Engineering"), (2, "Engineering"), (3, "Sales")]
dept_df = spark.createDataFrame(dept_data, ["id", "dept"])

# Force a broadcast join: dept_df is tiny, so it is shipped to all executors
joined = df.join(F.broadcast(dept_df), "id", "inner")
joined.display()

# Inspect the plan: look for "BroadcastHashJoin" (broadcast) vs "SortMergeJoin" (shuffle)
joined.explain()

# Tune the auto-broadcast threshold (e.g. allow broadcasts up to 50 MB)
# spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 50 * 1024 * 1024)
#
# Caution:
#   - Only broadcast genuinely small tables; a huge broadcast can OOM executors
#   - Spark's size estimate can be wrong on skewed/filtered data, which is why
#     F.broadcast() is useful when YOU know the table is small

In [0]:
data = [(1,["Python","SQL"]),
         (2,["Pyspark","AWS"])]

df = spark.createDataFrame(data,schema=("id","skills"))
display(df)

In [0]:
df = df.withColumn("skills",F.explode("skills"))

In [0]:
df1 = df.withColumn("fees",F.lit("NULL"))
display(df1)

In [0]:
df1.withColumn("fees", F.col("fees").try_cast("integer")).fillna({"fees": 0}).show()